# Kubernetes Assignments 1–5

Five progressive tasks, each building on the previous deployment:
1. Deploy a 3-node cluster + NGINX deployment (3 replicas)
2. Expose it via a NodePort service
3. Scale replicas to 5
4. Change the service type to ClusterIP
5. Add a second NGINX deployment + Ingress routing between two services

---
## Kubernetes Assignment 1
**Tasks:** Deploy a Kubernetes cluster for 3 nodes; create an NGINX deployment of 3 replicas.

In [1]:
cluster_cmds = '''
# Local 3-node cluster with Minikube
minikube start --nodes 3 --driver=docker
kubectl get nodes
'''
print(cluster_cmds)



# Local 3-node cluster with Minikube
minikube start --nodes 3 --driver=docker
kubectl get nodes



In [2]:
import os
os.makedirs('project/k8s', exist_ok=True)

with open('project/k8s/01-nginx-deployment.yaml', 'w') as f:
    f.write('''apiVersion: apps/v1
kind: Deployment
metadata:
  name: nginx-deployment
  labels:
    app: nginx
spec:
  replicas: 3
  selector:
    matchLabels:
      app: nginx
  template:
    metadata:
      labels:
        app: nginx
    spec:
      containers:
        - name: nginx
          image: nginx:latest
          ports:
            - containerPort: 80
''')
print("Created project/k8s/01-nginx-deployment.yaml")
print("Apply: kubectl apply -f project/k8s/01-nginx-deployment.yaml")
print("Verify: kubectl get deployments,pods -o wide")


Created project/k8s/01-nginx-deployment.yaml
Apply: kubectl apply -f project/k8s/01-nginx-deployment.yaml
Verify: kubectl get deployments,pods -o wide


---
## Kubernetes Assignment 2
**Tasks:** Use the previous deployment; create a NodePort service for the NGINX deployment; verify it in a browser.

In [3]:
with open('project/k8s/02-nginx-nodeport-service.yaml', 'w') as f:
    f.write('''apiVersion: v1
kind: Service
metadata:
  name: nginx-nodeport-svc
spec:
  type: NodePort
  selector:
    app: nginx
  ports:
    - port: 80
      targetPort: 80
      nodePort: 30080
''')
print("Created project/k8s/02-nginx-nodeport-service.yaml")
print("Apply: kubectl apply -f project/k8s/02-nginx-nodeport-service.yaml")
print("Verify in browser: minikube service nginx-nodeport-svc --url")


Created project/k8s/02-nginx-nodeport-service.yaml
Apply: kubectl apply -f project/k8s/02-nginx-nodeport-service.yaml
Verify in browser: minikube service nginx-nodeport-svc --url


---
## Kubernetes Assignment 3
**Tasks:** Use the previous deployment; change the replicas to 5.

In [4]:
scale_cmds = '''
kubectl scale deployment nginx-deployment --replicas=5
kubectl get pods -l app=nginx
'''
print(scale_cmds)

# Equivalent declarative change (recommended so the YAML stays the source of truth)
with open('project/k8s/01-nginx-deployment.yaml') as f:
    content = f.read()
content = content.replace('replicas: 3', 'replicas: 5')
with open('project/k8s/03-nginx-deployment-5replicas.yaml', 'w') as f:
    f.write(content)
print("Created project/k8s/03-nginx-deployment-5replicas.yaml")
print("Apply: kubectl apply -f project/k8s/03-nginx-deployment-5replicas.yaml")



kubectl scale deployment nginx-deployment --replicas=5
kubectl get pods -l app=nginx

Created project/k8s/03-nginx-deployment-5replicas.yaml
Apply: kubectl apply -f project/k8s/03-nginx-deployment-5replicas.yaml


---
## Kubernetes Assignment 4
**Tasks:** Use the previous deployment; change the service type to ClusterIP.

In [5]:
with open('project/k8s/04-nginx-clusterip-service.yaml', 'w') as f:
    f.write('''apiVersion: v1
kind: Service
metadata:
  name: nginx-clusterip-svc
spec:
  type: ClusterIP
  selector:
    app: nginx
  ports:
    - port: 80
      targetPort: 80
''')
print("Created project/k8s/04-nginx-clusterip-service.yaml")
print("Apply: kubectl apply -f project/k8s/04-nginx-clusterip-service.yaml")
print("Verify from inside the cluster (ClusterIP is not reachable externally):")
print("  kubectl run tmp-curl --rm -it --image=curlimages/curl -- curl http://nginx-clusterip-svc")


Created project/k8s/04-nginx-clusterip-service.yaml
Apply: kubectl apply -f project/k8s/04-nginx-clusterip-service.yaml
Verify from inside the cluster (ClusterIP is not reachable externally):
  kubectl run tmp-curl --rm -it --image=curlimages/curl -- curl http://nginx-clusterip-svc


---
## Kubernetes Assignment 5
**Tasks:** Use the previous deployment; deploy an NGINX deployment of 3 replicas; create an NGINX service of type ClusterIP; create an Ingress routing NGINX-to-NGINX (i.e. two NGINX-based services fronted by one Ingress, on different paths).

In [6]:
import os

# Second NGINX-based "app" (re-using the nginx image but with a distinct label to simulate
# a second service, e.g. an "Apache-like" backend) so the Ingress has two real targets.
with open('project/k8s/05a-nginx2-deployment.yaml', 'w') as f:
    f.write('''apiVersion: apps/v1
kind: Deployment
metadata:
  name: nginx2-deployment
  labels:
    app: nginx2
spec:
  replicas: 3
  selector:
    matchLabels:
      app: nginx2
  template:
    metadata:
      labels:
        app: nginx2
    spec:
      containers:
        - name: nginx2
          image: nginx:latest
          ports:
            - containerPort: 80
''')

with open('project/k8s/05b-nginx2-clusterip-service.yaml', 'w') as f:
    f.write('''apiVersion: v1
kind: Service
metadata:
  name: nginx2-clusterip-svc
spec:
  type: ClusterIP
  selector:
    app: nginx2
  ports:
    - port: 80
      targetPort: 80
''')

with open('project/k8s/05c-ingress.yaml', 'w') as f:
    f.write('''apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: nginx-ingress
  annotations:
    nginx.ingress.kubernetes.io/rewrite-target: /
spec:
  ingressClassName: nginx
  rules:
    - host: demo.local
      http:
        paths:
          - path: /app1
            pathType: Prefix
            backend:
              service:
                name: nginx-clusterip-svc
                port:
                  number: 80
          - path: /app2
            pathType: Prefix
            backend:
              service:
                name: nginx2-clusterip-svc
                port:
                  number: 80
''')
print("Created project/k8s/05a-nginx2-deployment.yaml")
print("Created project/k8s/05b-nginx2-clusterip-service.yaml")
print("Created project/k8s/05c-ingress.yaml")
print()
print("Enable the ingress controller (Minikube):  minikube addons enable ingress")
print("Apply everything: kubectl apply -f project/k8s/")
print("Add to /etc/hosts:  <minikube ip>  demo.local")
print("Test:  curl http://demo.local/app1   (routes to nginx-clusterip-svc)")
print("       curl http://demo.local/app2   (routes to nginx2-clusterip-svc)")


Created project/k8s/05a-nginx2-deployment.yaml
Created project/k8s/05b-nginx2-clusterip-service.yaml
Created project/k8s/05c-ingress.yaml

Enable the ingress controller (Minikube):  minikube addons enable ingress
Apply everything: kubectl apply -f project/k8s/
Add to /etc/hosts:  <minikube ip>  demo.local
Test:  curl http://demo.local/app1   (routes to nginx-clusterip-svc)
       curl http://demo.local/app2   (routes to nginx2-clusterip-svc)
